# Structured Streaming — 03: Stateful Aggregations & Windowing

This notebook covers three core stateful patterns in Structured Streaming:

## Three Types of Streaming Aggregations
- **Tumbling windows** — fixed, non-overlapping intervals; every event belongs to exactly one window.
- **Sliding windows** — overlapping intervals; one event can appear in multiple windows.
- **Session windows** — gap-based grouping; windows close when no event arrives within a defined gap.

## Watermarking
- Watermarks tell Spark how late data can arrive before a window is finalised and its state discarded.
- Syntax: `.withWatermark("event_time", "30 seconds")`
- Without a watermark the streaming state grows unboundedly — every window stays open forever.
- With a watermark, Spark drops windows older than `max_seen_event_time - watermark_delay`.

## Output Modes for Aggregations
- **complete** — re-emits ALL groups / windows every trigger; requires an aggregation.
- **update** — emits only the groups that changed in the current micro-batch.
- **append** — emits a window only AFTER its watermark has passed (window is finalised); requires watermark.
- `append` without watermark is **not allowed** for aggregations.

## Event Time vs Processing Time
- **Event time** — the timestamp embedded in the data itself (when the event actually happened).
- **Processing time** — the wall-clock time when Spark processes the event (current_timestamp).
- Windowing on event time correctly handles out-of-order and late-arriving data.
- Windowing on processing time is simpler but cannot recover from latency or replay.

## Setup

In [ ]:
import tempfile, os, time, threading, json, shutil
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, window, count, sum as spark_sum, avg, max as spark_max,
    current_timestamp, expr, lit, rand, when
)
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, IntegerType

spark = (
    SparkSession.builder
    .appName("SS-03-Stateful")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

## 1. Tumbling Window — Non-Overlapping Fixed Intervals

A **tumbling window** divides time into non-overlapping buckets of equal size.
Every event belongs to **exactly one** window — there is no overlap.

Syntax:
```python
window(timeCol, windowDuration)  # e.g. window("event_time", "10 seconds")
```

**Useful for:**
- Per-minute event counts
- Hourly revenue aggregations
- Daily totals

**Output mode trade-off:**
- `outputMode("complete")` — re-emits **all** windows every trigger (safe, more data written)
- `outputMode("update")` — emits only **changed** windows (less data, but sink must handle upserts)

In [ ]:
BASE = tempfile.mkdtemp().replace('\\', '/') + '/ss03'
os.makedirs(BASE, exist_ok=True)

CKPT1 = BASE + '/ckpt_tumbling'

rate_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
    .withColumn("event_time", current_timestamp())
    .withColumn("category", when(col("value") % 3 == 0, lit("A"))
                             .when(col("value") % 3 == 1, lit("B"))
                             .otherwise(lit("C")))
    .withColumn("amount", (col("value") % 100 + 1).cast("double"))
)

windowed = (
    rate_df
    .withWatermark("event_time", "5 seconds")
    .groupBy(window(col("event_time"), "10 seconds"), col("category"))
    .agg(count("*").alias("event_count"), spark_sum("amount").alias("total_amount"))
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("category"),
        col("event_count"),
        col("total_amount")
    )
)

q_tumbling = (
    windowed
    .writeStream
    .format("memory")
    .queryName("tumbling_windows")
    .outputMode("complete")
    .option("checkpointLocation", CKPT1)
    .start()
)

print("Tumbling window query running. Waiting 30 seconds...")
time.sleep(30)
print("\nTumbling window results:")
spark.sql("SELECT * FROM tumbling_windows ORDER BY window_start, category").show(20, truncate=False)
q_tumbling.stop()

## 2. Sliding Window — Overlapping Intervals

A **sliding window** fires every `slideDuration` seconds but each window covers the last `windowDuration`.
A single event can belong to **multiple overlapping windows**.

Syntax:
```python
window(timeCol, windowDuration, slideDuration)
# e.g. window("event_time", "20 seconds", "5 seconds")
# => a new window opens every 5s, each covering a 20s span
```

**Useful for:**
- 5-minute rolling averages updated every 1 minute
- Moving sums / exponential smoothing approximations

**Trade-off vs tumbling:**
- More windows = more state per trigger = more memory pressure
- Ratio of overlap: `windowDuration / slideDuration` windows open at any time
- If `windowDuration == slideDuration` you get a tumbling window

In [ ]:
CKPT2 = BASE + '/ckpt_sliding'

sliding_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 4)
    .load()
    .withColumn("event_time", current_timestamp())
    .withColumn("amount", (col("value") % 50 + 1).cast("double"))
)

sliding_agg = (
    sliding_df
    .withWatermark("event_time", "10 seconds")
    .groupBy(window(col("event_time"), "20 seconds", "5 seconds"))
    .agg(
        count("*").alias("events"),
        avg("amount").alias("avg_amount"),
        spark_max("amount").alias("max_amount")
    )
    .select(
        col("window.start").alias("win_start"),
        col("window.end").alias("win_end"),
        col("events"),
        col("avg_amount"),
        col("max_amount")
    )
)

q_sliding = (
    sliding_agg
    .writeStream
    .format("memory")
    .queryName("sliding_windows")
    .outputMode("complete")
    .option("checkpointLocation", CKPT2)
    .start()
)

print("Sliding window query running. Waiting 35 seconds...")
time.sleep(35)
print("\nSliding window results (sorted by win_start):")
spark.sql("SELECT * FROM sliding_windows ORDER BY win_start").show(20, truncate=False)
q_sliding.stop()

## 3. Watermarking — Handling Late Data

### What is a Watermark?
A **watermark** tells Spark: *"I expect data to be at most X late.
If an event arrives later than that, drop it."*

Without a watermark, Spark must keep **every window open forever** — state grows unboundedly.
With a watermark, Spark finalises and **discards** windows older than:
```
max_seen_event_time  -  watermark_delay
```

### Syntax
```python
.withWatermark("event_time", "30 seconds")
```

### Watermark Threshold Trade-off
| Threshold | Accuracy for late data | Memory usage |
|---|---|---|
| Higher (e.g. 5 min) | Better — more late events accepted | Higher — state held longer |
| Lower (e.g. 5 sec) | Lower — late events dropped sooner | Lower — state cleaned quickly |

### Output Mode Interaction
- `outputMode("append")` — a window is emitted **only after** its watermark has passed (finalised).
  No partial results; you get exactly one row per window.
- `outputMode("update")` — partial updates arrive as data flows in; final update when watermark passes.

In [ ]:
CKPT3 = BASE + '/ckpt_watermark'

wm_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 3)
    .load()
    .withColumn("event_time", current_timestamp())
    .withColumn("value_sq", col("value") * col("value"))
)

# With append output mode: windows appear only after watermark delay passes
wm_agg = (
    wm_df
    .withWatermark("event_time", "10 seconds")
    .groupBy(window(col("event_time"), "10 seconds"))
    .agg(count("*").alias("cnt"), spark_sum("value_sq").alias("sum_sq"))
    .select(
        col("window.start").alias("win_start"),
        col("window.end").alias("win_end"),
        col("cnt"),
        col("sum_sq")
    )
)

q_wm = (
    wm_agg
    .writeStream
    .format("memory")
    .queryName("watermark_demo")
    .outputMode("append")    # windows only appear once finalised
    .option("checkpointLocation", CKPT3)
    .start()
)

print("Watermark + append mode. Waiting 40 seconds for windows to finalise...")
time.sleep(40)
print("\nFinalised windows (append mode — only complete windows appear):")
spark.sql("SELECT * FROM watermark_demo ORDER BY win_start").show(truncate=False)
q_wm.stop()

## 4. Stateful groupBy — Running Counts Without Windows

Not every stateful aggregation needs a time window.
A plain `groupBy().count()` with `outputMode("complete")` maintains a **running total per key**
across **all events since the stream started**.

State is bounded only by the **cardinality of the key** — if you group by `category` with 4 values,
Spark keeps exactly 4 state entries, regardless of how many events have arrived.

This is safe for **low-cardinality keys** (e.g. status codes, categories, country codes).
High-cardinality keys (e.g. user_id with millions of users) will accumulate unbounded state
unless you add a watermark or use `mapGroupsWithState` / `flatMapGroupsWithState` with TTL.

In [ ]:
CKPT4 = BASE + '/ckpt_running'

running_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 6)
    .load()
    .withColumn("category", when(col("value") % 4 == 0, lit("W"))
                             .when(col("value") % 4 == 1, lit("X"))
                             .when(col("value") % 4 == 2, lit("Y"))
                             .otherwise(lit("Z")))
)

running_count = (
    running_df
    .groupBy("category")
    .count()
    .orderBy("category")
)

q_running = (
    running_count
    .writeStream
    .format("memory")
    .queryName("running_counts")
    .outputMode("complete")
    .option("checkpointLocation", CKPT4)
    .start()
)

print("Running count stream. Polling 4 times with 5s sleep:")
for i in range(4):
    time.sleep(5)
    print(f'\n--- Poll {i+1} (t={(i+1)*5}s) ---')
    spark.sql("SELECT * FROM running_counts ORDER BY category").show()

q_running.stop()

## 5. Update vs Complete vs Append — When to Use Each

| Output Mode | When to use | Requirements |
|---|---|---|
| `append` | Raw events, finalised windows | No agg, OR agg **with watermark** |
| `complete` | Running totals, small cardinality groups | **Aggregation required** |
| `update` | Changed rows per trigger | Aggregation; sink must support upserts |

### Sink Compatibility
| Sink | append | complete | update |
|---|---|---|---|
| `console` | Yes | Yes | Yes |
| `memory` | Yes | Yes | Yes |
| `file` (parquet/csv/json) | Yes | No | No |
| Delta Lake | Yes | Yes | Yes |
| Kafka | Yes | No | Yes |

**Key rule:** if you have an aggregation without a watermark, `append` mode is **not allowed**
because Spark cannot know when a group is finalised — it raises an `AnalysisException`.

## 6. Inspecting State — Event Time Progress

Every `StreamingQuery` exposes a `.lastProgress` dict that includes an `"eventTime"` sub-dict:

```json
{
  "eventTime": {
    "avg": "2024-01-01T12:00:00.000Z",
    "max": "2024-01-01T12:00:05.000Z",
    "min": "2024-01-01T11:59:55.000Z",
    "watermark": "2024-01-01T11:59:55.000Z"
  }
}
```

- `max` — the most recent event time seen in the current micro-batch.
- `watermark` — how far the watermark has advanced; windows older than this are finalised.
- The gap `max - watermark` should roughly equal your watermark delay.

Watching the watermark advance is the best way to verify that your watermark configuration
is working correctly and that state is being cleaned up as expected.

In [ ]:
CKPT5 = BASE + '/ckpt_inspect'

inspect_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 2)
    .load()
    .withColumn("event_time", current_timestamp())
)

q_inspect = (
    inspect_df
    .withWatermark("event_time", "10 seconds")
    .groupBy(window(col("event_time"), "10 seconds"))
    .agg(count("*").alias("n"))
    .writeStream
    .format("memory")
    .queryName("inspect_stream")
    .outputMode("update")
    .option("checkpointLocation", CKPT5)
    .start()
)

for i in range(3):
    time.sleep(8)
    p = q_inspect.lastProgress
    if p and p.get("eventTime"):
        print(f"Poll {i+1} — eventTime: {p['eventTime']}")
    else:
        print(f"Poll {i+1} — no eventTime yet")

q_inspect.stop()

## Key Takeaways

- **Tumbling windows**: no overlap, each event in exactly one window.
  Use `window(timeCol, windowDuration)` — simplest and cheapest stateful pattern.

- **Sliding windows**: overlap, each event in multiple windows.
  Use `window(timeCol, windowDuration, slideDuration)` — more expressive but higher state cost.
  At any moment `windowDuration / slideDuration` windows are open simultaneously.

- **Watermarks bound state size** — always add a watermark for production aggregations.
  Without one, state (open windows) grows without bound and will eventually OOM.

- **complete mode** re-emits all groups each trigger — safe but writes more data.
  **update mode** emits only changed groups — efficient but requires an upsert-capable sink.
  **append mode** emits only finalised windows — requires a watermark; ideal for file/Delta sinks.

- **Running `groupBy().count()` without watermark**: state is unbounded — only safe for
  low-cardinality keys (a handful of distinct values, not millions of user IDs).

## Practice Exercises

1. Change the tumbling window from `"10 seconds"` to `"30 seconds"` — do you see fewer rows
   in the result? Why does a longer window produce fewer but larger aggregates?

2. Add a `having count(*) > 5` equivalent by chaining `.filter(col("event_count") > 5)`
   after the `groupBy` in the tumbling window query. Does it work on streams?

3. Remove the `.withWatermark(...)` call from the sliding window query and change
   `outputMode` to `"complete"`. Run for 60 seconds and compare memory usage
   (observe `q_sliding.lastProgress['stateOperators']`).

4. Try `session_window(col("event_time"), gapDuration="5 seconds")` (Spark 3.2+)
   in place of `window(...)` in the tumbling query.
   Observe how session windows cluster bursts of activity rather than using fixed-size buckets.

## Teardown

In [ ]:
shutil.rmtree(BASE, ignore_errors=True)
print("Temp directories cleaned up.")
spark.stop()
print("SparkSession stopped.")